# Silver - ecommerce_rastreamento_entregas

Desenvolvido por: Ygor Moraes

Este notebook lê a Bronze `ecommerce_rastreamento_entregas` e grava a Silver tratada em Delta.

Regras aplicadas:
- converter IDs para inteiro e `dt_evento` para timestamp;
- normalizar `status_entrega` e manter apenas status válidos;
- deduplicar eventos por `id_rastreamento`;
- calcular `dias_em_transito` por pedido (primeiro evento válido do pedido até último evento válido do pedido);
- manter particionamento por `ano` e `mes`.

Golds/KPIs alimentadas:
- tempo médio mensal de entrega;
- SLA mensal de entregas;
- problemas mensais em entregas;
- entregas em feriados;
- volume de entregas por estado.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Silver de rastreamento.

from pyspark.sql.functions import (
    col,
    count,
    when,
    lower,
    trim,
    regexp_replace,
    translate,
    to_timestamp,
    current_timestamp,
    row_number,
    min as spark_min,
    max as spark_max,
    datediff,
    to_date
)

from pyspark.sql.window import Window

BRONZE_TABLE = "ecommerce_rastreamento_entregas"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "ecommerce_rastreamento_entregas"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

KEY_COLUMNS = ["id_rastreamento"]

SILVER_WRITE_MODE = "overwrite"

STATUS_VALIDOS = [
    "coletado",
    "em separacao",
    "em transito",
    "saiu para entrega",
    "entregue"
]

BRONZE_REQUIRED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "codigo_rastreio",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao",
    "bronze_source_file",
    "bronze_ingested_at",
    "ano",
    "mes"
]

SILVER_REQUIRED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "codigo_rastreio",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao",
    "dias_em_transito",
    "bronze_source_file",
    "bronze_ingested_at",
    "silver_processed_at",
    "ano",
    "mes"
]

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Bronze: {BRONZE_PATH}")
print(f"Destino Silver: {SILVER_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Bronze e valida se as colunas necessárias existem.

df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

bronze_columns = df_bronze.columns

missing_bronze_columns = [
    c for c in BRONZE_REQUIRED_COLUMNS
    if c not in bronze_columns
]

if missing_bronze_columns:
    raise Exception(f"Colunas obrigatórias ausentes na Bronze: {missing_bronze_columns}")

total_bronze = df_bronze.count()

print("Bronze lida com sucesso.")
print(f"Total de registros na Bronze: {total_bronze}")

df_bronze.printSchema()

In [0]:
# Valida se IDs e dt_evento podem ser convertidos antes da transformação.

df_validacao_conversoes = (
    df_bronze
    .withColumn("id_rastreamento_int", col("id_rastreamento").cast("int"))
    .withColumn("id_pedido_ecommerce_int", col("id_pedido_ecommerce").cast("int"))
    .withColumn("id_transportadora_int", col("id_transportadora").cast("int"))
    .withColumn("dt_evento_ts", to_timestamp(col("dt_evento")))
    .select(
        count("*").alias("total_linhas"),

        count(
            when(
                col("id_rastreamento").isNotNull()
                & col("id_rastreamento_int").isNull(),
                True
            )
        ).alias("falhas_id_rastreamento"),

        count(
            when(
                col("id_pedido_ecommerce").isNotNull()
                & col("id_pedido_ecommerce_int").isNull(),
                True
            )
        ).alias("falhas_id_pedido_ecommerce"),

        count(
            when(
                col("id_transportadora").isNotNull()
                & col("id_transportadora_int").isNull(),
                True
            )
        ).alias("falhas_id_transportadora"),

        count(
            when(
                col("dt_evento").isNotNull()
                & col("dt_evento_ts").isNull(),
                True
            )
        ).alias("falhas_dt_evento")
    )
)

display(df_validacao_conversoes)

validacao_conversoes = df_validacao_conversoes.collect()[0]

if validacao_conversoes["falhas_id_rastreamento"] > 0:
    raise Exception("Existem valores de id_rastreamento que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_id_pedido_ecommerce"] > 0:
    raise Exception("Existem valores de id_pedido_ecommerce que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_id_transportadora"] > 0:
    raise Exception("Existem valores de id_transportadora que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_dt_evento"] > 0:
    raise Exception("Existem valores de dt_evento que não podem ser convertidos para timestamp.")

print("Validação OK: conversões principais podem ser feitas.")

In [0]:
# Normaliza status_entrega e identifica registros válidos para a Silver.

df_bronze_status = (
    df_bronze
    .withColumn(
        "status_entrega_normalizado",
        regexp_replace(
            translate(
                lower(trim(col("status_entrega"))),
                "áàâãéêíóôõúç",
                "aaaaeeiooouc"
            ),
            "\\s+",
            " "
        )
    )
    .withColumn(
        "status_valido",
        col("status_entrega_normalizado").isin(STATUS_VALIDOS)
    )
)

df_validacao_status = (
    df_bronze_status
    .groupBy("status_entrega_normalizado", "status_valido")
    .count()
    .orderBy("status_valido", "status_entrega_normalizado")
)

display(df_validacao_status)

total_status_invalidos = (
    df_bronze_status
    .filter(~col("status_valido"))
    .count()
)

print(f"Total de registros com status fora do fluxo: {total_status_invalidos}")
print("Registros com status fora do fluxo ficarão apenas na Bronze.")

In [0]:
# Aplica casts, filtra status válidos e deduplica por id_rastreamento.

df_silver_base = (
    df_bronze_status
    .withColumn("id_rastreamento_int", col("id_rastreamento").cast("int"))
    .withColumn("id_pedido_ecommerce_int", col("id_pedido_ecommerce").cast("int"))
    .withColumn("id_transportadora_int", col("id_transportadora").cast("int"))
    .withColumn("dt_evento_ts", to_timestamp(col("dt_evento")))
    .filter(col("status_valido"))
    .filter(col("id_rastreamento_int").isNotNull())
)

window_deduplicacao = (
    Window
    .partitionBy("id_rastreamento_int")
    .orderBy(
        col("dt_evento_ts").desc_nulls_last(),
        col("bronze_ingested_at").desc_nulls_last()
    )
)

df_silver_deduplicada = (
    df_silver_base
    .withColumn("rn", row_number().over(window_deduplicacao))
    .filter(col("rn") == 1)
)

# Calcula dias_em_transito por pedido usando o primeiro e o último evento válido.
# Essa métrica será usada nas Golds de tempo médio e SLA.

window_pedido = Window.partitionBy("id_pedido_ecommerce_int")

df_silver_com_datas = (
    df_silver_deduplicada
    .withColumn(
        "dt_primeiro_evento_pedido",
        spark_min(col("dt_evento_ts")).over(window_pedido)
    )
    .withColumn(
        "dt_ultimo_evento_pedido",
        spark_max(col("dt_evento_ts")).over(window_pedido)
    )
    .withColumn(
        "dias_em_transito",
        datediff(
            to_date(col("dt_ultimo_evento_pedido")),
            to_date(col("dt_primeiro_evento_pedido"))
        )
    )
)

df_silver = (
    df_silver_com_datas
    .select(
        col("id_rastreamento_int").alias("id_rastreamento"),
        col("id_pedido_ecommerce_int").alias("id_pedido_ecommerce"),
        trim(col("codigo_rastreio")).alias("codigo_rastreio"),
        col("id_transportadora_int").alias("id_transportadora"),
        col("status_entrega_normalizado").alias("status_entrega"),
        col("dt_evento_ts").alias("dt_evento"),
        trim(col("observacao")).alias("observacao"),
        col("dias_em_transito"),

        col("bronze_source_file").cast("string").alias("bronze_source_file"),
        col("bronze_ingested_at").cast("timestamp").alias("bronze_ingested_at"),

        current_timestamp().alias("silver_processed_at"),

        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes")
    )
)

total_silver = df_silver.count()

print("Silver criada em memória.")
print(f"Total de registros na Silver: {total_silver}")

df_silver.printSchema()

In [0]:
# Valida deduplicação, status e campos críticos antes da gravação.

total_ids_distintos_validos = (
    df_bronze_status
    .filter(col("status_valido"))
    .filter(col("id_rastreamento").isNotNull())
    .select(col("id_rastreamento").cast("int").alias("id_rastreamento"))
    .distinct()
    .count()
)

duplicados_silver = (
    df_silver
    .groupBy("id_rastreamento")
    .count()
    .filter(col("count") > 1)
    .count()
)

status_invalidos_silver = (
    df_silver
    .filter(~col("status_entrega").isin(STATUS_VALIDOS))
    .count()
)

df_validacao_silver = df_silver.select(
    count("*").alias("total_linhas"),

    count(when(col("id_rastreamento").isNull(), True)).alias("id_rastreamento_nulo"),
    count(when(col("id_pedido_ecommerce").isNull(), True)).alias("id_pedido_ecommerce_nulo"),
    count(when(col("id_transportadora").isNull(), True)).alias("id_transportadora_nulo"),

    count(when(col("status_entrega").isNull(), True)).alias("status_entrega_nulo"),
    count(when(col("dt_evento").isNull(), True)).alias("dt_evento_nulo"),

    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),

    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo"),

    count(when(col("dias_em_transito").isNull(), True)).alias("dias_em_transito_nulo"),
    count(
        when(
            col("dias_em_transito").isNotNull()
            & (col("dias_em_transito") < 0),
            True
        )
    ).alias("dias_em_transito_negativo")
)

display(df_validacao_silver)

print(f"Total Bronze: {total_bronze}")
print(f"Total Silver após filtro e deduplicação: {total_silver}")
print(f"IDs distintos válidos na Bronze: {total_ids_distintos_validos}")
print(f"Duplicados na Silver: {duplicados_silver}")
print(f"Status inválidos na Silver: {status_invalidos_silver}")

if total_silver != total_ids_distintos_validos:
    raise Exception("Erro: quantidade da Silver diferente dos IDs válidos distintos da Bronze.")

if duplicados_silver > 0:
    raise Exception("Erro: ainda existem id_rastreamento duplicados na Silver.")

if status_invalidos_silver > 0:
    raise Exception("Erro: existem status fora do fluxo na Silver.")

validacao_silver = df_validacao_silver.collect()[0]

if validacao_silver["id_rastreamento_nulo"] > 0:
    raise Exception("Erro: existem registros com id_rastreamento nulo na Silver.")

if validacao_silver["id_pedido_ecommerce_nulo"] > 0:
    raise Exception("Erro: existem registros com id_pedido_ecommerce nulo na Silver.")

if validacao_silver["id_transportadora_nulo"] > 0:
    raise Exception("Erro: existem registros com id_transportadora nulo na Silver.")

if validacao_silver["status_entrega_nulo"] > 0:
    raise Exception("Erro: existem registros com status_entrega nulo na Silver.")

if validacao_silver["dt_evento_nulo"] > 0:
    raise Exception("Erro: existem registros com dt_evento nulo na Silver.")

if validacao_silver["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Silver.")

if validacao_silver["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Silver.")

if validacao_silver["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

if validacao_silver["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_silver["silver_processed_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem silver_processed_at.")

if validacao_silver["dias_em_transito_negativo"] > 0:
    raise Exception("Erro: existem registros com dias_em_transito negativo.")

print("Validação OK: Silver em memória aprovada.")
print("Observação: dias_em_transito foi mantido como diferença entre primeiro e último evento válido do pedido.")

In [0]:
# Grava a Silver em Delta com overwrite.

(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print(f"Silver gravada com sucesso em Delta: {SILVER_PATH}")
print(f"Modo de escrita utilizado: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Silver gravada e valida volume, duplicidade e schema final.

df_silver_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

total_silver_saved = df_silver_saved.count()

duplicados_silver_saved = (
    df_silver_saved
    .groupBy("id_rastreamento")
    .count()
    .filter(col("count") > 1)
    .count()
)

status_invalidos_saved = (
    df_silver_saved
    .filter(~col("status_entrega").isin(STATUS_VALIDOS))
    .count()
)

colunas_silver_saved = df_silver_saved.columns

colunas_ausentes = [
    c for c in SILVER_REQUIRED_COLUMNS
    if c not in colunas_silver_saved
]

print(f"IDs distintos válidos na Bronze: {total_ids_distintos_validos}")
print(f"Total Silver gravada: {total_silver_saved}")
print(f"IDs duplicados na Silver gravada: {duplicados_silver_saved}")
print(f"Status inválidos na Silver gravada: {status_invalidos_saved}")

if total_silver_saved != total_ids_distintos_validos:
    raise Exception("Erro: quantidade da Silver gravada diferente dos IDs válidos distintos da Bronze.")

if duplicados_silver_saved > 0:
    raise Exception("Erro: existem id_rastreamento duplicados na Silver gravada.")

if status_invalidos_saved > 0:
    raise Exception("Erro: existem status fora do fluxo na Silver gravada.")

if colunas_ausentes:
    raise Exception(f"Erro: colunas obrigatórias ausentes na Silver: {colunas_ausentes}")

df_silver_saved.printSchema()

print("Validação final da Silver OK.")